# Module 04-1: 문서 요약 에이전트 구현

## 학습 목표

1. **State 설계**: 7개 필드로 구성된 `DocumentSummaryState`를 TypedDict로 정의한다
2. **4노드 파이프라인**: fetch_document → analyze_content → generate_summary → format_output 흐름을 구현한다
3. **조건부 에러 분기**: `_route_after_fetch`, `_route_after_analyze`로 에러 발생 시 조기 종료한다
4. **functools.partial 의존성 주입**: LLM을 노드 함수에 외부에서 주입하여 테스트 가능하게 만든다
5. **FakeLLM E2E 테스트**: 실제 API 없이 전체 파이프라인을 테스트한다

---
> 참고 문서: `docs/part2-first-agent/04-building-first-agent.md`

## 개념 요약

### 문서 요약 에이전트 워크플로우

```
[시작]
  |
  v
[fetch_document] ─── 문서 로드 및 유효성 검사
  |
  ├─ (문서가 비어있음) ──> [END] (에러 경로)
  |
  v
[analyze_content] ─── LLM으로 핵심 내용 분석
  |
  ├─ (분석 실패) ──> [END] (에러 경로)
  |
  v
[generate_summary] ─── LLM으로 요약 생성
  |
  v
[format_output] ─── 결과 포맷팅 후 완료
  |
  v
[END]
```

### functools.partial로 의존성 주입

```python
from functools import partial

# 테스트 시: FakeLLM 주입
graph.add_node("analyze_content", partial(analyze_content, llm=fake_llm))

# 운영 시: 실제 LLM 주입
graph.add_node("analyze_content", partial(analyze_content, llm=real_llm))
```

### State 필드 역할

| 필드 | 타입 | 채우는 노드 | 설명 |
|------|------|------------|------|
| `source_text` | `str` | 사용자 입력 | 원본 텍스트 |
| `document` | `dict\|None` | fetch_document | 파싱된 문서 정보 |
| `analysis` | `dict\|None` | analyze_content | 분석 결과 |
| `summary` | `str\|None` | generate_summary | 요약문 |
| `final_output` | `str\|None` | format_output | 포맷된 결과 |
| `current_step` | `str` | 모든 노드 | 진행 상태 |
| `error` | `str\|None` | 에러 노드 | 에러 메시지 |

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

import json
from typing import TypedDict
from functools import partial
from langgraph.graph import StateGraph, END
from common.fake_llm import FakeLLM

print("환경 설정 완료")

## 실습 1: DocumentSummaryState 정의

문서 요약 에이전트의 상태를 정의하세요.

**필드 목록**:
- `source_text`: 원본 문서 텍스트 (str)
- `document`: 로드된 문서 정보, fetch_document가 채움 (dict | None)
- `analysis`: 분석 결과, analyze_content가 채움 (dict | None)
- `summary`: 생성된 요약문 (str | None)
- `final_output`: 최종 포맷팅된 결과 (str | None)
- `current_step`: 현재 진행 단계 (str)
- `error`: 에러 메시지 (str | None)

In [ ]:
# TODO: 여기에 코드를 작성하세요

# 힌트 1: TypedDict를 상속하여 DocumentSummaryState를 정의합니다
# 힌트 2: Optional 필드는 `str | None` 또는 `dict | None` 타입을 사용합니다
# 힌트 3:
# class DocumentSummaryState(TypedDict):
#     source_text: str
#     document: dict | None
#     analysis: dict | None
#     summary: str | None
#     final_output: str | None
#     current_step: str
#     error: str | None

# DocumentSummaryState를 정의하세요

print("State 정의 완료")

In [ ]:
# 검증 셀
assert 'DocumentSummaryState' in dir(), "DocumentSummaryState가 정의되어야 합니다"
fields = list(DocumentSummaryState.__annotations__.keys())
required_fields = ['source_text', 'document', 'analysis', 'summary', 'final_output', 'current_step', 'error']
for f in required_fields:
    assert f in fields, f"'{f}' 필드가 없습니다. 현재 필드: {fields}"
print(f"필드 목록: {fields}")
print("✅ 실습 1 완료!")

## 실습 2: fetch_document 노드 구현

문서를 로드하고 기본 정보를 추출하는 노드를 구현하세요.

**동작 규칙**:
- `source_text`가 비어있으면 `error` 필드에 메시지를 설정하고 `current_step = "error"` 반환
- 정상이면 `document` 딕셔너리(`title`, `content`, `word_count`, `line_count`)와 `current_step = "fetched"` 반환

In [ ]:
# TODO: 여기에 코드를 작성하세요

# 힌트 1: source_text가 비어있으면 error 경로로 분기합니다
# 힌트 2: lines = source_text.strip().split('\n')으로 줄을 분리합니다
# 힌트 3:
# def fetch_document(state: DocumentSummaryState) -> dict:
#     source_text = state["source_text"]
#     if not source_text or not source_text.strip():
#         return {"error": "문서가 비어있습니다.", "current_step": "error"}
#     lines = source_text.strip().split("\n")
#     title = lines[0] if lines else "제목 없음"
#     content = "\n".join(lines[1:]).strip() if len(lines) > 1 else source_text
#     word_count = len(source_text.split())
#     document = {"title": title, "content": content, "word_count": word_count, "line_count": len(lines)}
#     return {"document": document, "current_step": "fetched"}

# fetch_document 함수를 구현하세요

print("fetch_document 구현 완료")

In [ ]:
# 검증 셀
base_state = {"source_text": "", "document": None, "analysis": None,
              "summary": None, "final_output": None, "current_step": "start", "error": None}

# 정상 케이스
r1 = fetch_document({**base_state, "source_text": "Python 가이드\n파이썬은 간결한 언어입니다."})
assert r1["current_step"] == "fetched", f"정상 케이스에서 current_step이 'fetched'여야 합니다"
assert r1["document"]["title"] == "Python 가이드", f"title이 'Python 가이드'여야 합니다"
assert r1["document"]["word_count"] > 0, "word_count가 0보다 커야 합니다"

# 에러 케이스
r2 = fetch_document({**base_state, "source_text": ""})
assert r2.get("error") is not None, "빈 문서에서 error 필드가 설정되어야 합니다"
assert r2["current_step"] == "error", "에러 시 current_step이 'error'여야 합니다"

print(f"정상 결과: title='{r1['document']['title']}', word_count={r1['document']['word_count']}")
print(f"에러 결과: {r2['error']}")
print("✅ 실습 2 완료!")

## 실습 3: analyze_content 노드 구현

LLM을 사용하여 문서의 핵심 내용을 분석하는 노드를 구현하세요.

**동작 규칙**:
- `document`가 없으면 에러 반환
- `llm`이 None이면 에러 반환
- LLM에 JSON 형식 분석을 요청하고, JSON 파싱 실패 시 텍스트 기반 분석 결과 반환
- 성공 시 `analysis` 딕셔너리와 `current_step = "analyzed"` 반환

In [ ]:
# TODO: 여기에 코드를 작성하세요

# 힌트 1: llm=None 기본값으로 정의하여 의존성 주입을 허용합니다
# 힌트 2: llm.invoke(prompt).content로 LLM 응답을 가져옵니다
# 힌트 3:
# def analyze_content(state: DocumentSummaryState, llm=None) -> dict:
#     document = state.get("document")
#     if not document:
#         return {"error": "문서가 로드되지 않았습니다.", "current_step": "error"}
#     if llm is None:
#         return {"error": "LLM이 설정되지 않았습니다.", "current_step": "error"}
#     prompt = f"다음 문서를 분석해주세요.\n\n제목: {document['title']}\n내용: {document['content']}\n\nJSON으로: {{\"keywords\": [...], \"topic\": \"...\", \"difficulty\": \"초급|중급|고급\"}}"
#     try:
#         response = llm.invoke(prompt)
#         content = response.content
#         try:
#             analysis = json.loads(content)
#         except json.JSONDecodeError:
#             analysis = {"keywords": [], "topic": content[:100], "difficulty": "중급", "raw_response": content}
#         return {"analysis": analysis, "current_step": "analyzed"}
#     except Exception as e:
#         return {"error": f"분석 중 오류: {str(e)}", "current_step": "error"}

# analyze_content 함수를 구현하세요

print("analyze_content 구현 완료")

In [ ]:
# 검증 셀
fake_llm_for_test = FakeLLM(
    responses={
        "분석": json.dumps({"keywords": ["Python", "FastAPI"], "topic": "웹 개발", "difficulty": "중급"}, ensure_ascii=False)
    },
    default_response="분석 완료"
)
base_state = {"source_text": "", "document": {"title": "Python 가이드", "content": "분석 대상 텍스트"},
              "analysis": None, "summary": None, "final_output": None, "current_step": "fetched", "error": None}

analyze_fn = partial(analyze_content, llm=fake_llm_for_test)
r = analyze_fn(base_state)
assert r["current_step"] == "analyzed", f"current_step이 'analyzed'여야 합니다. 현재: {r.get('current_step')}"
assert r["analysis"] is not None, "analysis 필드가 설정되어야 합니다"

# LLM 없을 때 에러
r2 = analyze_content(base_state, llm=None)
assert r2.get("error") is not None, "llm=None일 때 error가 설정되어야 합니다"

print(f"분석 결과: {r['analysis']}")
print("✅ 실습 3 완료!")

## 실습 4: generate_summary + format_output 노드 구현

**generate_summary**: LLM으로 문서를 요약하는 노드
- `document`와 `analysis` 상태를 읽어 LLM에 요약 요청
- 성공 시 `summary`와 `current_step = "summarized"` 반환

**format_output**: 최종 결과를 포맷팅하는 노드 (LLM 불필요)
- `document`, `analysis`, `summary`를 읽어 보기 좋은 출력 구성
- 성공 시 `final_output`과 `current_step = "completed"` 반환

In [ ]:
# TODO: 여기에 코드를 작성하세요

# 힌트 1: generate_summary는 analyze_content와 유사한 패턴으로 구현합니다
# 힌트 2: format_output은 LLM 없이 상태 값을 조합하여 출력을 구성합니다
# 힌트 3 (generate_summary):
# def generate_summary(state: DocumentSummaryState, llm=None) -> dict:
#     if llm is None:
#         return {"error": "LLM이 설정되지 않았습니다.", "current_step": "error"}
#     document = state.get("document", {})
#     analysis = state.get("analysis", {})
#     prompt = f"다음 문서를 3줄 이내로 요약해주세요.\n제목: {document.get('title', 'N/A')}\n내용: {document.get('content', 'N/A')}\n분석: {json.dumps(analysis, ensure_ascii=False)}"
#     try:
#         response = llm.invoke(prompt)
#         return {"summary": response.content, "current_step": "summarized"}
#     except Exception as e:
#         return {"error": f"요약 실패: {str(e)}", "current_step": "error"}

# generate_summary, format_output 함수를 구현하세요

print("generate_summary, format_output 구현 완료")

In [ ]:
# 검증 셀
fake_llm2 = FakeLLM(responses={"요약": "FastAPI는 고성능 Python 웹 프레임워크입니다."})
test_state = {
    "source_text": "FastAPI 시작하기",
    "document": {"title": "FastAPI 시작하기", "content": "FastAPI는 Python 웹 프레임워크입니다.", "word_count": 6, "line_count": 1},
    "analysis": {"keywords": ["Python", "FastAPI"], "topic": "웹 개발", "difficulty": "중급"},
    "summary": None, "final_output": None, "current_step": "analyzed", "error": None
}

# generate_summary 테스트
sum_fn = partial(generate_summary, llm=fake_llm2)
r_sum = sum_fn(test_state)
assert r_sum["current_step"] == "summarized", f"current_step이 'summarized'여야 합니다. 현재: {r_sum.get('current_step')}"
assert r_sum["summary"] is not None, "summary 필드가 설정되어야 합니다"

# format_output 테스트
test_state["summary"] = r_sum["summary"]
r_fmt = format_output(test_state)
assert r_fmt["current_step"] == "completed", f"current_step이 'completed'여야 합니다. 현재: {r_fmt.get('current_step')}"
assert r_fmt["final_output"] is not None, "final_output 필드가 설정되어야 합니다"
assert len(r_fmt["final_output"]) > 0, "final_output이 비어있지 않아야 합니다"

print(f"요약: {r_sum['summary']}")
print(f"최종 출력 (앞 100자): {r_fmt['final_output'][:100]}")
print("✅ 실습 4 완료!")

## 실습 5: 조건부 분기 함수 및 그래프 조립

**분기 함수** 구현:
- `_route_after_fetch(state)`: error 있으면 `"error"`, 없으면 `"analyze"` 반환
- `_route_after_analyze(state)`: error 있으면 `"error"`, 없으면 `"summarize"` 반환

**build_summarizer_graph(llm)** 함수 구현:
- 4개 노드 등록 (LLM 필요 노드는 partial로 주입)
- 조건부 엣지 2개 + 일반 엣지 2개 연결

In [ ]:
# TODO: 여기에 코드를 작성하세요

# 힌트 1: _route_after_fetch와 _route_after_analyze는 state.get("error")로 분기합니다
# 힌트 2: build_summarizer_graph에서 partial(analyze_content, llm=llm)으로 LLM을 주입합니다
# 힌트 3:
# def _route_after_fetch(state: DocumentSummaryState) -> str:
#     return "error" if state.get("error") else "analyze"
#
# def _route_after_analyze(state: DocumentSummaryState) -> str:
#     return "error" if state.get("error") else "summarize"
#
# def build_summarizer_graph(llm=None):
#     graph = StateGraph(DocumentSummaryState)
#     graph.add_node("fetch_document", fetch_document)
#     graph.add_node("analyze_content", partial(analyze_content, llm=llm))
#     graph.add_node("generate_summary", partial(generate_summary, llm=llm))
#     graph.add_node("format_output", format_output)
#     graph.set_entry_point("fetch_document")
#     graph.add_conditional_edges("fetch_document", _route_after_fetch, {"analyze": "analyze_content", "error": END})
#     graph.add_conditional_edges("analyze_content", _route_after_analyze, {"summarize": "generate_summary", "error": END})
#     graph.add_edge("generate_summary", "format_output")
#     graph.add_edge("format_output", END)
#     return graph.compile()

# _route_after_fetch, _route_after_analyze, build_summarizer_graph를 구현하세요

print("그래프 조립 함수 정의 완료")

In [ ]:
# 검증 셀: 분기 함수 테스트
s_ok = {"source_text": "x", "document": None, "analysis": None, "summary": None,
        "final_output": None, "current_step": "fetched", "error": None}
s_err = {**s_ok, "error": "에러 발생"}

assert _route_after_fetch(s_ok) == "analyze", "error 없을 때 'analyze'를 반환해야 합니다"
assert _route_after_fetch(s_err) == "error", "error 있을 때 'error'를 반환해야 합니다"
assert _route_after_analyze(s_ok) == "summarize", "error 없을 때 'summarize'를 반환해야 합니다"
assert _route_after_analyze(s_err) == "error", "error 있을 때 'error'를 반환해야 합니다"

print("✅ 분기 함수 검증 완료!")

## 실습 6: FakeLLM E2E 테스트

FakeLLM을 설정하고 그래프를 빌드하여 정상 경로와 에러 경로를 모두 테스트합니다.

In [ ]:
# FakeLLM 설정 및 그래프 빌드
fake_llm = FakeLLM(
    responses={
        "분석": json.dumps({
            "keywords": ["Python", "FastAPI", "웹 프레임워크"],
            "topic": "Python 웹 개발",
            "difficulty": "중급",
        }, ensure_ascii=False),
        "요약": (
            "본 문서는 FastAPI를 활용한 Python 웹 개발 방법을 설명합니다. "
            "REST API 설계 원칙과 비동기 처리 패턴을 중심으로 다룹니다."
        ),
    },
    default_response="요청을 처리했습니다.",
)

graph = build_summarizer_graph(llm=fake_llm)
print("그래프 빌드 완료!")

In [ ]:
# 정상 경로 테스트
initial_state = {
    "source_text": (
        "FastAPI 시작하기\n"
        "FastAPI는 Python 3.7+로 API를 구축하기 위한 고성능 웹 프레임워크입니다. "
        "Starlette과 Pydantic을 기반으로 하며, 자동 문서 생성과 타입 검사를 제공합니다."
    ),
    "document": None, "analysis": None, "summary": None,
    "final_output": None, "current_step": "start", "error": None,
}

print("=" * 60)
print("  문서 요약 에이전트 실행 (정상 경로)")
print("=" * 60)

events = []
for event in graph.stream(initial_state):
    events.append(event)
    node_name = list(event.keys())[0]
    print(f">> 노드 실행 완료: {node_name}")

# 최종 상태 수집
final_state = {}
for event in events:
    for node_output in event.values():
        final_state.update(node_output)

if final_state.get("final_output"):
    print("\n" + final_state["final_output"])
elif final_state.get("error"):
    print(f"\n[에러] {final_state['error']}")

In [ ]:
# 에러 경로 테스트 (빈 문서)
error_state = {
    "source_text": "",
    "document": None, "analysis": None, "summary": None,
    "final_output": None, "current_step": "start", "error": None,
}

print("=" * 60)
print("  에러 케이스: 빈 문서 테스트")
print("=" * 60)

for event in graph.stream(error_state):
    node_name = list(event.keys())[0]
    node_output = event[node_name]
    print(f">> 노드: {node_name}")
    if node_output.get("error"):
        print(f"   에러 발생: {node_output['error']}")
    print(f"   현재 단계: {node_output.get('current_step', 'N/A')}")

In [ ]:
# E2E 검증 셀
# 정상 경로 검증
result_normal = graph.invoke(initial_state)
assert result_normal.get("final_output") is not None, "정상 경로에서 final_output이 설정되어야 합니다"
assert result_normal.get("current_step") == "completed", f"정상 경로에서 current_step이 'completed'여야 합니다. 현재: {result_normal.get('current_step')}"
assert result_normal.get("error") is None, "정상 경로에서 error가 없어야 합니다"

# 에러 경로 검증
result_error = graph.invoke(error_state)
assert result_error.get("error") is not None, "에러 경로에서 error 필드가 설정되어야 합니다"
assert result_error.get("current_step") == "error", "에러 경로에서 current_step이 'error'여야 합니다"
assert result_error.get("final_output") is None, "에러 경로에서 final_output이 None이어야 합니다"

print("✅ 실습 6 완료! E2E 테스트 성공")

## 정리

### 오늘 배운 내용
- **7필드 State 설계**: source_text → document → analysis → summary → final_output + current_step + error
- **4노드 파이프라인**: fetch → analyze → summarize → format
- **조건부 에러 분기**: `add_conditional_edges`로 에러 시 조기 종료
- **functools.partial**: LLM을 외부에서 주입하여 테스트 가능한 아키텍처 구현
- **FakeLLM E2E 테스트**: 실제 API 없이 전체 파이프라인 검증

### 다음 노트북 안내
**02_email_classifier.ipynb** - 이메일 분류 에이전트를 직접 설계하고 구현합니다.